**GemmaSAM Colab Jupyter Notebook**

Copy this notebook into Google Colab to run

In [ ]:
!git clone https://github.com/xiaod83/GemmaSAM.git
%cd /content/GemmaSAM

#Clone the repository and cd into the folder

In [ ]:
!pip -q install -U torch transformers numpy opencv-python Pillow tqdym matplotlib monai sam2

#Download necessary dependencies 
#Downloading manually because requirements.txt had issues with compatibility
#Ignore errors of gradio and pillow incompatibility, does not affect operation

In [ ]:
!pip -q install -U "huggingface_hub"

from huggingface_hub import snapshot_download
from huggingface_hub import hf_hub_download

mllm_dir = snapshot_download(repo_id = "google/gemma-4-E4b-it")
medsam2_dir = hf_hub_download(repo_id = "wanglab/MedSAM2", filename="MedSAM2_latest.pt")

print("MLLM dir:", mllm_dir)

#Download Gemma-4-E4b-it from huggingface
#Download MedSAM2 from huggingface

In [ ]:
%cd /content/GemmaSAM/infer

import os
import glob

os.environ['MEDSAM2_DIR'] = medsam2_dir
os.environ['MLLM_DIR'] = mllm_dir

# Fix Triton looking for CUDA 13 NVRTC by symlinking the existing CUDA 12 library
nvrtc_dir = '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib'
cu12_libs = glob.glob(f'{nvrtc_dir}/libnvrtc-builtins.so.12*')
cu13_lib = f'{nvrtc_dir}/libnvrtc-builtins.so.13.0'
if cu12_libs and not os.path.exists(cu13_lib):
    os.symlink(cu12_libs[0], cu13_lib)

# Ensure the directory is in LD_LIBRARY_PATH
if os.path.exists(nvrtc_dir):
    os.environ['LD_LIBRARY_PATH'] = nvrtc_dir + ':' + os.environ.get('LD_LIBRARY_PATH', '')

#The above code is because of issues with Triton in Google Colab


'''
Important: img_path is the path to the image you want to segment, target-description is the prompt you wish to give the mllm; do not change model-path or seg-checkpoint
'''

!python run_single_inference.py \
    --img_path demo/path/sample_medical_image.png \ 
    --target-description "prompt" \ 
    --model-path "$MLLM_DIR" \
    --seg-checkpoint "$MEDSAM2_DIR" \ 